# Step 13 — Multi-Agent

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role, and wire both into a `Crew`. You'll run the same two agents under two different `Process` strategies: `sequential` — the same fixed pipeline this repo's full demo project (`src/research_crew/crew.py`) uses, just defined directly in this notebook instead of via `@CrewBase` and YAML config files — and `hierarchical`, where a manager decides at runtime which agent handles each task, instead of that being fixed in code.

## Learning objective

By the end of this notebook, you will:

- Understand why a second, complementary agent can produce something neither agent produces alone
- Have chained two `Task`s together with `context=[...]`, so one agent's output feeds directly into another's input
- Have run your first real `Crew` with two agents and `process=Process.sequential`
- Understand how `process=Process.hierarchical` differs: task order is still fixed, but *which* agent handles each task is a manager's runtime decision, not something wired into the code
- Have run the same two agents under a manager, via `manager_llm` (or a custom `manager_agent`)

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent unchanged
- The same `.env` setup as the previous steps

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

## How this works

Two agents and two tasks, wired into a `Crew` with `process=Process.sequential`:

1. **`researcher`** is Step 09's exact same Researcher agent, unchanged.
2. **`analyst`** is a new agent with a different `role`/`goal`/`backstory` — someone who turns raw findings into a report for a specific audience, a job the Researcher was never asked to do.
3. **`research_task`** is assigned to the Researcher; **`analysis_task`** is assigned to the Analyst, with one crucial addition: `context=[research_task]`. This tells CrewAI to feed the first task's output into the second automatically — the same idea as Step 06's chain prompting, but wired declaratively instead of copy-pasted by hand.
4. **`crew.kickoff()`** runs both tasks in the order they appear in `tasks=[...]`, because `process=Process.sequential`.

Everything else in the cell (the `tracing=True` flag and the `crewai_event_bus.flush()` call afterward) is optional bookkeeping that prints a shareable trace link — it doesn't affect what the agents do.

Further down, the same two agents run again under `process=Process.hierarchical` — a different way of wiring the same pieces together.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Task 1: research — assigned to the Researcher ─────────────────────────────
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
    agent=researcher,
)

# ── Task 2: analysis — assigned to the Analyst, chained to Task 1 via `context` ─
analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
    agent=analyst,
    context=[research_task],
)

# ── Crew — same sequential process as src/research_crew/crew.py ──────────────
crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Researcher output ===")
print(research_task.output.raw)
print("\n=== Analyst output ===")
print(analysis_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d1f215ec-5841-48e1-abca-807768187ef8                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: 9a4fd57c-6b1f-45e0-bfa7-f5fd8ed5cfa5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the            │
│  risk-based framework. The Act establishes a tiered regulatory environment where the intensity of obligations   │
│  is directly proportional to the potential harm posed by the AI system.                                         │
│                                                                                                                 │
│  ### Part 1: The Four Risk-Based Categories                                                                     │
│                                                                                                                 │
│  The EU AI Act classifies AI systems into four tiers, moving from prohibited to effectively unregulated.        │
│                                                                                                                 │
│  1.  **Unacceptable Risk (Prohibited):**                                                                        │
│      *   **Definition:** Systems considered a clear threat to fundamental rights and safety.                    │
│      *   **Examples:** Cognitive behavioral manipulation, social scoring by governments, real-time remote       │
│  biometric identification in public spaces for law enforcement (with narrow exceptions), and indiscriminate     │
│  scraping of facial images from the internet or CCTV.                                                           │
│      *   **Outcome:** These systems are strictly banned from the EU market.                                     │
│                                                                                                                 │
│  2.  **High-Risk (Strictly Regulated):**                                                                        │
│      *   **Definition:** Systems that pose significant risks to health, safety, or fundamental rights. This     │
│  includes AI used in critical infrastructure, education, employment, essential private/public services, law     │
│  enforcement, and migration management.                                                                         │
│      *   **Outcome:** Mandatory compliance with strict requirements, human oversight, and conformity            │
│  assessments before market entry.                                                                               │
│                                                                                                                 │
│  3.  **Limited Risk (Transparency Obligations):**                                                               │
│      *   **Definition:** Systems with specific transparency risks.                                              │
│      *   **Examples:** Chatbots (AI assistants) and emotion recognition systems.                                │
│      *   **Outcome:** Users must be informed that they are interacting with a machine, and content generated    │
│  by AI (like deepfakes) must be clearly labeled as synthetic.                                                   │
│                                                                                                                 │
│  4.  **Minimal Risk (No Regulation):**                                                                          │
│      *   **Definition:** The vast majority of AI system

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what obligations     │
│  apply to providers of high-risk AI systems.                                                                    │
│  Agent:                                                                                                         │
│  EU AI Act Senior Data Researcher                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Reporting Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **To:** Product Team                                                                                           │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Date:** May 22, 2024                                                                                         │
│  **Subject:** Compliance Requirements for AI-Powered Hiring Tools under the EU AI Act                           │
│                                                                                                                 │
│  ### Overview                                                                                                   │
│  Under the EU AI Act, AI systems used in **employment, workers management, and access to                        │
│  self-employment**—specifically those used for recruitment, screening, ranking, and evaluating candidates—are   │
│  explicitly classified as **High-Risk AI Systems**.                                                             │
│                                                                                                                 │
│  Because your product falls under this category, it is subject to rigorous regulatory obligations to ensure it  │
│  does not infringe upon the fundamental rights of job applicants.                                               │
│                                                                                                                 │
│  ### Mandatory Compliance Obligations                                                                           │
│  As a provider of a high-risk AI system, your team must integrate the following requirements into your          │
│  development lifecycle:                                                                                         │
│                                                                                                                 │
│  1.  **Risk Management:** Establish a continuous system to identify and mitigate foreseeable risks to           │
│  fundamental rights (e.g., bias in hiring algorithms) throughout the system’s lifecycle.                        │
│  2.  **Data Governance:** Ensure training, validation, and testing datasets are high-quality, representative,   │
│  and audited for bias to prevent discriminatory outcomes.                                                       │
│  3.  **Technical Documentation:** Maintain comprehensive documentation demonstrating compliance, which must be  │
│  readily available for national authorities.                                                                    │
│  4.  **Automatic Logging:** Enable automatic event recording to ensure full traceability of the system’s        │
│  decision-making processes.                                                                                     │
│  5.  **Transparency & User Info:** Provide clear "Instructions for Use" so recruiters (deployers) can           │
│  accurately interpret the system’s outputs and limitations.                                                     │
│  6.  **Human Oversight:** Design the UI/UX to ensure a natural person can oversee the AI’s recommendations and  │
│  intervene if necessary.                                                                                        │
│  7.  **Robustness & Security:** Ensure the system meets

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  EU AI Act Reporting Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d1f215ec-5841-48e1-abca-807768187ef8                                                                           │
│  Final Output: **To:** Product Team                                                                             │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Date:** May 22, 2024                                                                                         │
│  **Subject:** Compliance Requirements for AI-Powered Hiring Tools under the EU AI Act                           │
│                                                                                                                 │
│  ### Overview                                                                                                   │
│  Under the EU AI Act, AI systems used in **employment, workers management, and access to                        │
│  self-employment**—specifically those used for recruitment, screening, ranking, and evaluating candidates—are   │
│  explicitly classified as **High-Risk AI Systems**.                                                             │
│                                                                                                                 │
│  Because your product falls under this category, it is subject to rigorous regulatory obligations to ensure it  │
│  does not infringe upon the fundamental rights of job applicants.                                               │
│                                                                                                                 │
│  ### Mandatory Compliance Obligations                                                                           │
│  As a provider of a high-risk AI system, your team must integrate the following requirements into your          │
│  development lifecycle:                                                                                         │
│                                                                                                                 │
│  1.  **Risk Management:** Establish a continuous system to identify and mitigate foreseeable risks to           │
│  fundamental rights (e.g., bias in hiring algorithms) throughout the system’s lifecycle.                        │
│  2.  **Data Governance:** Ensure training, validation, and testing datasets are high-quality, representative,   │
│  and audited for bias to prevent discriminatory outcomes.                                                       │
│  3.  **Technical Documentation:** Maintain comprehensive documentation demonstrating compliance, which must be  │
│  readily available for national authorities.                                                                    │
│  4.  **Automatic Logging:** Enable automatic event recording to ensure full traceability of the system’s        │
│  decision-making processes.                                                                                     │
│  5.  **Transparency & User Info:** Provide clear "Instructions for Use" so recruiters (deployers) can           │
│  accurately interpret the system’s outputs and limitations.                                                     │
│  6.  **Human Oversight:** Design the UI/UX to ensure a

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 06e8faa6-4143-4fbe-9fb3-247f13beb90e                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/06e8faa6-4143-4fbe-9fb3-247f13beb90e?access_code=TRA │
│ CE-4d60f1c485                                                                                                   │
│ 🔑 Access Code: TRACE-4d60f1c485                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Researcher output ===
As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the risk-based framework. The Act establishes a tiered regulatory environment where the intensity of obligations is directly proportional to the potential harm posed by the AI system.

### Part 1: The Four Risk-Based Categories

The EU AI Act classifies AI systems into four tiers, moving from prohibited to effectively unregulated.

1.  **Unacceptable Risk (Prohibited):**
    *   **Definition:** Systems considered a clear threat to fundamental rights and safety.
    *   **Examples:** Cognitive behavioral manipulation, social scoring by governments, real-time remote biometric identification in public spaces for law enforcement (with narrow exceptions), and indiscriminate scraping of facial images from the internet or CCTV.
    *   **Outcome:** These systems are strictly banned from the EU market.

2.  **High-Risk (Strictly Regulated):**
    *   **Definition:** Systems

### A non-sequential pattern: manager delegation (`Process.hierarchical`)

Above, each `Task` is hard-wired to one `Agent` via `agent=...`, and `crew.kickoff()` runs them in the order they appear in `tasks=[...]`. `Process.hierarchical` removes that fixed wiring: the tasks below don't get an `agent=` at all. Instead, a **manager** — either an `Agent` CrewAI builds for you from `manager_llm`, or your own `Agent` passed as `manager_agent` — reads each task and decides, at that moment, which of the crew's agents is best suited, then delegates to them via a tool call.

Two things worth being precise about, since "hierarchical" invites the wrong mental model:

- **Task order still isn't dynamic.** The `for` loop over `tasks=[...]` doesn't change — task 1 still runs before task 2. What's dynamic is *who* does each task, decided fresh every run, not *when* it happens.
- **The manager is not one of your agents.** It's a separate role that does no task work itself — it only reads tasks and delegates. CrewAI enforces this: a `manager_agent` may not also appear in `agents=[...]`, and it may not be given `tools=` (only the agents it delegates to can have tools).

The `researcher` and `analyst` from above are reused unchanged below — same `role`/`goal`/`backstory`, no code changes to either agent. Only the `Crew`'s wiring is different.

In [2]:
# ── Same two tasks, but neither gets an `agent=` — the manager decides who does what ──
research_task_h = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
)

analysis_task_h = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
)

# ── Crew — same two agents as coworkers, orchestrated by a manager instead of fixed code ──
hierarchical_crew = Crew(
    agents=[researcher, analyst],  # the manager's coworkers — the manager itself is not one of them
    tasks=[research_task_h, analysis_task_h],
    process=Process.hierarchical,
    manager_llm=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"),
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes — same as
    # the sequential crew above, so you can compare both runs side by side.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result_h = hierarchical_crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Research task, as delegated by the manager ===")
print(research_task_h.output.raw)
print("\n=== Analysis task, as delegated by the manager ===")
print(analysis_task_h.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  b0951055-7cd0-469b-b074-99fd4eff1556                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: 923ea71b-effe-4f38-a2fa-a634f64ca6a3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': "The user wants a structured summary of the EU AI Act's risk-based categories                │
│  (unacceptable, high-risk, limited, minimal) and the specific obligations that apply to providers of high-risk  │
│  ...                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Please provide a detailed, structured summary of the EU AI Act's four risk categories: unacceptable,     │
│  high-risk, limited, and minimal. For the 'high-risk' category, clearly outline the specific obligations        │
│  imposed on providers. Ensure the information is accurate and easy to follow.                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **To:** Product Team                                                                                           │
│  **From:** Management                                                                                           │
│  **Subject:** Compliance Requirements and Timeline for AI-Powered Hiring Tool                                   │
│                                                                                                                 │
│  As our AI-powered hiring tool is classified as a **High-Risk AI system** under the EU AI Act (specifically     │
│  concerning AI in recruitment and employment), we are subject to the strictest tier of regulatory               │
│  requirements. To ensure we remain compliant, the following obligations must be integrated into our product     │
│  development lifecycle.                                                                                         │
│                                                                                                                 │
│  ### Mandatory Compliance Obligations                                                                           │
│  We must implement the following nine pillars of compliance to achieve conformity:                              │
│                                                                                                                 │
│  1.  **Risk Management System:** Establish a continuous, iterative process to identify and mitigate risks to    │
│  fundamental rights throughout the lifecycle.                                                                   │
│  2.  **Data Governance:** Ensure all training, validation, and testing datasets meet strict quality criteria,   │
│  focusing on relevance, representativeness, and freedom from bias.                                              │
│  3.  **Technical Documentation:** Maintain comprehensive, up-to-date documentation that demonstrates our        │
│  compliance to regulatory authorities.                                                                          │
│  4.  **Automatic Logging (Record-Keeping):** Design the system to automatically record operational events to    │
│  facilitate traceability and incident investigation.                                                            │
│  5.  **Transparency and Information:** Provide clear "Instructions for Use" to ensure deployers can properly    │
│  interpret and act on the system’s outputs.                                                                     │
│  6.  **Human Oversight:** Integrate an interface that allows human supervisors to effectively monitor,          │
│  intervene, or interrupt system operations.                                                                     │
│  7.  **Accuracy, Robustness, and Cybersecurity:** Ensure the system is resilient against errors, faults, and    │
│  unauthorized exploitation.                                                                                     │
│  8.  **Quality Management System (QMS):** Document internal procedures for design, testing, and continuous      │
│  monitoring.                                                                                                    │
│  9.  **Conformity Assessment & CE Marking:** Perform a mandatory conformity assessment and obtain the           │
│  necessary CE marking before the compliance deadline.  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  b0951055-7cd0-469b-b074-99fd4eff1556                                                                           │
│  Final Output: **To:** Product Team                                                                             │
│  **From:** Management                                                                                           │
│  **Subject:** Compliance Requirements and Timeline for AI-Powered Hiring Tool                                   │
│                                                                                                                 │
│  As our AI-powered hiring tool is classified as a **High-Risk AI system** under the EU AI Act (specifically     │
│  concerning AI in recruitment and employment), we are subject to the strictest tier of regulatory               │
│  requirements. To ensure we remain compliant, the following obligations must be integrated into our product     │
│  development lifecycle.                                                                                         │
│                                                                                                                 │
│  ### Mandatory Compliance Obligations                                                                           │
│  We must implement the following nine pillars of compliance to achieve conformity:                              │
│                                                                                                                 │
│  1.  **Risk Management System:** Establish a continuous, iterative process to identify and mitigate risks to    │
│  fundamental rights throughout the lifecycle.                                                                   │
│  2.  **Data Governance:** Ensure all training, validation, and testing datasets meet strict quality criteria,   │
│  focusing on relevance, representativeness, and freedom from bias.                                              │
│  3.  **Technical Documentation:** Maintain comprehensive, up-to-date documentation that demonstrates our        │
│  compliance to regulatory authorities.                                                                          │
│  4.  **Automatic Logging (Record-Keeping):** Design the system to automatically record operational events to    │
│  facilitate traceability and incident investigation.                                                            │
│  5.  **Transparency and Information:** Provide clear "Instructions for Use" to ensure deployers can properly    │
│  interpret and act on the system’s outputs.                                                                     │
│  6.  **Human Oversight:** Integrate an interface that allows human supervisors to effectively monitor,          │
│  intervene, or interrupt system operations.                                                                     │
│  7.  **Accuracy, Robustness, and Cybersecurity:** Ensure the system is resilient against errors, faults, and    │
│  unauthorized exploitation.                                                                                     │
│  8.  **Quality Management System (QMS):** Document internal procedures for design, testing, and continuous      │
│  monitoring.                                          

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 6fcbd3d5-da59-4560-b254-1f68d86920cf                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/6fcbd3d5-da59-4560-b254-1f68d86920cf?access_code=TRA │
│ CE-bdd78e5cce                                                                                                   │
│ 🔑 Access Code: TRACE-bdd78e5cce                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Research task, as delegated by the manager ===
### 1. The EU AI Act: Four-Tier Risk Classification

The legislation employs a risk-based approach, mapping regulatory requirements to the potential harm a system may cause to fundamental rights and safety.

*   **Unacceptable Risk (Prohibited):**
    These systems are deemed incompatible with EU values. They are strictly prohibited from being placed on the market or put into service. This includes systems that use subliminal techniques to distort behavior, exploit vulnerabilities of specific groups (age, disability), engage in government-led social scoring, or conduct indiscriminate real-time remote biometric identification in public spaces for law enforcement (subject to limited, strictly defined exceptions).

*   **High-Risk (Strictly Regulated):**
    These are AI systems used in contexts where they could negatively impact safety or fundamental rights. Examples include AI in critical infrastructure, education (e.g., grading), emplo

## Your task

1. Run the sequential cell. Compare the Analyst's report to Step 09's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: remove `context=[research_task]` from `analysis_task` and rerun. Without that link, does the Analyst still somehow reference the Researcher's specific findings, or does it write a generic report from scratch?

3. Run the hierarchical cell. With `verbose=True`, the console output shows the manager's own reasoning and its "Delegate work to coworker" / "Ask question to coworker" tool calls — find the line where it picks the Researcher for the first task and the Analyst for the second. Compare `result_h` to `result` from the sequential run: same two jobs, same two agents — did the manager's routing produce a meaningfully different outcome, or basically the same one with extra steps (and extra LLM calls) in between? Both cells print a trace link to app.crewai.com (no signup needed) — open both and compare the timeline view side by side to see the delegation calls visually, not just in the console log.

4. Now swap in your own team's topic — keep the Researcher agent from Step 09, and give it a second, genuinely complementary role and task suited to your topic.

5. This is where your project shifts from guided exercises to your own design: use everything from Steps 03–13 to build and document your own agent. Fill in `REPORT.md` — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full report structure and what's expected. Make sure the **Sprint Progression** table has all five rows filled in — it's the fastest way for your reader to see the arc before they read the rest of the report.

## Shortcomings

The sequential pipeline is fixed: the Researcher always runs first, the Analyst always runs second, and the order never changes based on what either agent actually finds. `Process.hierarchical` fixes *who* handles each task, not *when* — the task order in `tasks=[...]` is still fixed, and it costs more: every task now takes an extra LLM call (the manager's own reasoning) on top of whichever agent it delegates to, and the manager's choice of delegate isn't guaranteed to be consistent between runs. For two agents with clearly non-overlapping roles like these, that extra machinery mostly buys you nothing — it starts to pay off once you have enough agents, or similar-enough roles, that hard-coding `agent=...` on every task stops being obvious.

Neither `Crew` above uses any of the tools/MCP/RAG grounding from Steps 10–12 — combining multi-agent orchestration with a grounded, tool-using Researcher is a natural next experiment, just not one this notebook walks you through.

This is the last step in the exercise series. From here, the main [README's use-case table](../../README.md#use-cases-to-pick-from) and `REPORT.md` ask you to step back and judge, for your own topic, which combination of everything covered — single vs. multi-agent, sequential vs. hierarchical, tools, MCP, RAG — is actually worth the added complexity.

## Resources for further reading

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`, covered briefly in Step 02

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. The hierarchical crew is the more natural fit for this: add the agent to `agents=[...]` and a `Task` for it to `tasks=[...]`, but don't pin it with `agent=...` — let the manager decide on its own whether and when to bring it in, rather than wiring it into a fixed spot in the pipeline like you'd have to for the sequential version. Does the output meaningfully change? If it doesn't, ask yourself why.

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.